# BS baseline — CatBoost по dNBR (был `fast_basr.ipynb`)

**Модуль:** `models/burn_severity` (сущность BS, степень поражения гари).
**Происхождение:** черновик сокомандника, переименован и перенесён в модуль без изменения логики.

## Что здесь есть

1. `rle_encode` — кодирование маски в RLE по правилам кейса.
2. `extract_features` — попиксельные признаки `dNBR`, `NBR_pre`, `NBR_post`
   из каналов B8A (индекс 6) и B12 (индекс 8) снимков Sentinel-2 до и после пожара.
3. `load_chip` / `train_baseline` — сборка датасета (262144 строки на чип) и обучение
   CatBoost MultiClass на 4 класса: 0 — нет гари, 1/2/3 — слабое/среднее/сильное поражение.
4. `predict_and_make_submission` — инференс и запись CSV.

## Как это связано с продакшн-кодом

Notebook — исследовательский черновик и остаётся справочным. Рабочий путь — модуль:

| Notebook | Модуль |
|---|---|
| `extract_features` | `models/burn_severity/predict.py::extract_features` (та же формула, без pandas) |
| `load_chip` | `models/burn_severity/predict.py::resolve_chip_paths` + `read_optical` (пути задаются аргументом, а не константой) |
| `train_baseline` | `python -m models.burn_severity.train --data-dir ...` |
| `predict_and_make_submission` | `python -m models.burn_severity.run_inference --data-dir ... --sample ...` |
| `rle_encode` | `models/common/rle.py::encode_rle` (общий для AF и BS) |

Отличия модуля от notebook: пути передаются аргументами, маска валидируется
(uint8, 512x512, только классы 0-3), CSV пишется через общий контракт и проверяется
на дубликаты, пересечения классов и каноничность RLE. Сабмишн отдельного модуля
собирается в `models/burn_severity/results/`, финальный — `scripts/collect_results.py`.

## Ограничения черновика

- Индексы каналов 6 и 8 взяты из документации кейса и на реальных GeoTIFF не проверены.
- Имя файла маски (`{chip_id}_mask.tif`) — предположение.
- Признаков всего три: пространственный контекст не используется, поэтому границы
  классов будут шумными; ближайшие улучшения — окрестностные статистики, NDVI/NDWI,
  SAR-каналы и landcover.
- Веса `.cbm` в репозиторий пока не добавлены: `models/burn_severity/artifacts/` пуст.


In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
import glob
import os

# --- 1. ФУНКЦИЯ КОДИРОВАНИЯ RLE (СТРОГО ПО ПРАВИЛАМ КЕЙСА) ---
def rle_encode(mask):
    """
    Кодирует бинарную маску (2D numpy array) в строку RLE.
    Пиксели нумеруются с 1, слева-направо, сверху-вниз.
    Формат: "старт длина старт длина ..."
    """
    pixels = mask.flatten()
    # Добавляем нули по краям, чтобы поймать границы серий
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2] # вычисляем длины
    return ' '.join(str(x) for x in runs)

# --- 2. ФИЧЕИНЖИНИРИНГ (Тот самый dNBR) ---
def extract_features(pre_img, post_img, landcover_img=None):
    """
    превращает 3D тензоры (C, 512, 512) в 2D таблицу (262144, N_features)
    pre_img / post_img: предполагаем каналы (по доке):
    idx 6 = B8A, idx 8 = B12
    """
    # Считаем индексы, добавляем 1e-8 от деления на ноль
    b8a_pre, b12_pre = pre_img[6], pre_img[8]
    b8a_post, b12_post = post_img[6], post_img[8]

    nbr_pre = (b8a_pre - b12_pre) / (b8a_pre + b12_pre + 1e-8)
    nbr_post = (b8a_post - b12_post) / (b8a_post + b12_post + 1e-8)
    dnbr = nbr_pre - nbr_post

    # Собираем DataFrame для CatBoost
    df = pd.DataFrame({
        'dNBR': dnbr.flatten(),
        'NBR_pre': nbr_pre.flatten(),
        'NBR_post': nbr_post.flatten()
    })

    # Если есть маска ландшафта - добавляем (поможет раскидать классы 1,2,3)
    if landcover_img is not None:
        df['landcover'] = landcover_img.flatten()

    return df

In [2]:
# Заглушка для функции чтения tif (напиши свою с rasterio/tifffile)
import os
import rasterio
import numpy as np

def load_chip(chip_id, data_dir='data', mode='train'):
    """
    Читает TIF-файлы для конкретного чипа.
    mode: 'train' (возвращает pre, post, mask) или 'test' (только pre, post)
    """
    # ВАЖНО: Подставь сюда свою реальную структуру папок.
    # Предполагаем, что файлы лежат как: data/train/bs/{chip_id}_pre.tif
    base_path = os.path.join(data_dir, mode, 'bs')

    # Пути к оптическим снимкам
    pre_path = os.path.join(base_path, f"{chip_id}_pre.tif")
    post_path = os.path.join(base_path, f"{chip_id}_post.tif")

    # 1. Читаем оптику Sentinel-2 (L2A)
    # rasterio автоматически вернет массив формы [10, 512, 512]
    with rasterio.open(pre_path) as src:
        pre_img = src.read().astype(np.float32)

    with rasterio.open(post_path) as src:
        post_img = src.read().astype(np.float32)

    # 2. Читаем маску (только для трейна)
    mask = None
    if mode == 'train':
        mask_path = os.path.join(base_path, f"{chip_id}_mask.tif") # имя файла маски может отличаться!
        with rasterio.open(mask_path) as src:
            # Читаем 1-й канал, чтобы получить матрицу [512, 512], а не [1, 512, 512]
            mask = src.read(1).astype(np.uint8)

    # --- БОНУС: Подключение радара и покрова (раскомментируй, когда понадобятся) ---
    # sar_pre_path = os.path.join(base_path, f"{chip_id}_sar_pre.tif")
    # with rasterio.open(sar_pre_path) as src:
    #     sar_pre = src.read().astype(np.float32) # [2, 512, 512] (каналы VV, VH)

    # aux_path = os.path.join(base_path, f"{chip_id}_aux.tif")
    # with rasterio.open(aux_path) as src:
    #     aux_data = src.read()
    #     # Допустим, канал 4 в aux_data - это тип покрова (landcover)
    #     # landcover = aux_data[3]

    return pre_img, post_img, mask

def train_baseline(train_chip_ids):
    X_list = []
    y_list = []

    print("Сборка датасета...")
    for chip_id in train_chip_ids[:30]: # Возьми для бейзлайна 30 чипов, чтобы проверить пайплайн
        pre, post, mask = load_chip(chip_id, 'train')

        # Получаем фичи
        X_df = extract_features(pre, post)

        # Разворачиваем маску в 1D (262144,)
        y = mask.flatten()

        X_list.append(X_df)
        y_list.append(y)

    X_train = pd.concat(X_list, ignore_index=True)
    y_train = np.concatenate(y_list)

    print("Обучение CatBoost...")
    # Обучаем быстро, итераций мало, чтобы тиммейт не ждал
    model = CatBoostClassifier(
        iterations=150,
        learning_rate=0.1,
        depth=6,
        loss_function='MultiClass', # Классы 0, 1, 2, 3
        task_type='CPU',
        verbose=25
    )
    # Если добавил landcover, укажи cat_features=['landcover']
    model.fit(X_train, y_train)
    model.save_model('baseline_bs_model.cbm')
    return model

In [5]:
def predict_and_make_submission(test_chip_ids, model_path='baseline_bs_model.cbm'):
    model = CatBoostClassifier()
    model.load_model(model_path)

    submission_rows = []

    print("Инференс на тесте...")
    for chip_id in test_chip_ids:
        # Для теста маски нет
        pre, post, _ = load_chip(chip_id, 'test')
        X_test = extract_features(pre, post)

        # Предикт классов (выдаст 1D массив 262144,)
        preds_1d = model.predict(X_test).flatten()

        # Восстанавливаем картинку 512x512
        pred_mask = preds_1d.reshape((512, 512))

        # По правилам нужно закодировать каждый класс (1,2,3) отдельно!
        for class_id in [1, 2, 3]:
            # Создаем бинарную маску для конкретного класса
            binary_mask = (pred_mask == class_id).astype(np.uint8)

            # Кодируем в RLE
            rle_str = rle_encode(binary_mask)

            submission_rows.append({
                'chip_id': chip_id,
                'class_id': class_id,
                'rle': rle_str # Если пусто (нет такого класса на чипе), rle_encode вернет '' (пустую строку)
            })

    # Сохраняем в формат сабмита
    sub_df = pd.DataFrame(submission_rows)
    sub_df.to_csv('submission_module2.csv', index=False)
    print("Готово! Файл submission_module2.csv сохранен.")

    # Тиммейту для API можешь возвращать саму pred_mask
    return pred_mask